# 04 — Análise Exploratória de Dados (EDA)

Explora volume, gravidade, distribuição temporal, geográfica, tipos de acidente, pista, clima, traçado, rodovias e causas.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PASTA_PROJETO = "TCC_PRF"
PASTA_PROCESSED = os.path.join(PASTA_PROJETO, "data", "processed")
PASTA_OUTPUT = os.path.join(PASTA_PROJETO, "outputs", "eda")
os.makedirs(PASTA_OUTPUT, exist_ok=True)

df = pd.read_parquet(os.path.join(PASTA_PROCESSED, "prf_acidentes_silver_2020_2025.parquet"))
print(df.shape)

def salvar_grafico(nome):
    plt.tight_layout()
    plt.savefig(os.path.join(PASTA_OUTPUT, nome), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

In [ ]:
total_acidentes = len(df)
total_graves = int(df["acidente_grave"].sum())
total_fatais = int(df["acidente_fatal"].sum())

kpis = pd.DataFrame({
    "Indicador": [
        "Total de acidentes",
        "Acidentes graves/fatais",
        "Acidentes não graves",
        "Acidentes com vítima fatal",
        "Total de mortos",
        "Total de feridos graves",
        "Total de feridos leves",
        "Taxa de acidentes graves (%)",
        "Taxa de acidentes fatais (%)"
    ],
    "Valor": [
        total_acidentes,
        total_graves,
        total_acidentes-total_graves,
        total_fatais,
        int(df["mortos"].sum()),
        int(df["feridos_graves"].sum()),
        int(df["feridos_leves"].sum()),
        round(total_graves/total_acidentes*100, 2),
        round(total_fatais/total_acidentes*100, 2)
    ]
})
display(kpis)
kpis.to_csv(os.path.join(PASTA_OUTPUT, "kpis_gerais.csv"), index=False, encoding="utf-8-sig")

In [ ]:
acidentes_ano = (
    df.groupby("ano")
      .agg(acidentes=("id","count"), graves=("acidente_grave","sum"),
           fatais=("acidente_fatal","sum"), mortos=("mortos","sum"))
      .reset_index()
)
acidentes_ano["taxa_grave"] = acidentes_ano["graves"]/acidentes_ano["acidentes"]*100
acidentes_ano["taxa_fatal"] = acidentes_ano["fatais"]/acidentes_ano["acidentes"]*100
display(acidentes_ano)

plt.figure(figsize=(10,6))
plt.bar(acidentes_ano["ano"], acidentes_ano["acidentes"])
plt.title("Quantidade de acidentes por ano")
plt.xlabel("Ano"); plt.ylabel("Número de acidentes"); plt.grid(axis="y", alpha=0.3)
salvar_grafico("01_acidentes_por_ano.png")

In [ ]:
def resumo_categoria(coluna, minimo=0):
    t = (df.groupby(coluna)
           .agg(acidentes=("id","count"), graves=("acidente_grave","sum"), mortos=("mortos","sum"))
           .reset_index())
    t["taxa_grave"] = t["graves"]/t["acidentes"]*100
    if minimo:
        t = t[t["acidentes"] >= minimo]
    return t.sort_values("taxa_grave", ascending=False)

fase = resumo_categoria("fase_dia")
uf = resumo_categoria("uf")
tipo_acidente = resumo_categoria("tipo_acidente", minimo=500)
clima = resumo_categoria("condicao_metereologica")
tipo_pista = resumo_categoria("tipo_pista")
uso_solo = resumo_categoria("uso_solo")

display(fase)
display(uf.head(27))
display(tipo_acidente.head(20))
display(clima)
display(tipo_pista)
display(uso_solo)

In [ ]:
colunas_tracado = [c for c in df.columns if c.startswith("tracado_") and c != "tracado_via"]
resultado_tracado = []
for c in colunas_tracado:
    sub = df[df[c] == 1]
    resultado_tracado.append({
        "tracado": c,
        "acidentes": len(sub),
        "graves": int(sub["acidente_grave"].sum()),
        "taxa_grave": sub["acidente_grave"].mean()*100 if len(sub) else np.nan
    })
tracado = pd.DataFrame(resultado_tracado).sort_values("taxa_grave", ascending=False)
display(tracado)

In [ ]:
df_causas = df[df["ano"] >= 2021].copy()
causas = (df_causas.groupby("causa_acidente")
         .agg(acidentes=("id","count"), graves=("acidente_grave","sum"), mortos=("mortos","sum"))
         .reset_index())
causas["taxa_grave"] = causas["graves"]/causas["acidentes"]*100
causas = causas.sort_values("acidentes", ascending=False)
display(causas.head(30))

rodovias = (df.groupby("br")
            .agg(acidentes=("id","count"), graves=("acidente_grave","sum"), mortos=("mortos","sum"))
            .reset_index())
rodovias["taxa_grave"] = rodovias["graves"]/rodovias["acidentes"]*100
display(rodovias.sort_values("acidentes", ascending=False).head(20))

In [ ]:
hora = (df.groupby("hora")
        .agg(acidentes=("id","count"), graves=("acidente_grave","sum"))
        .reset_index())
hora["taxa_grave"] = hora["graves"]/hora["acidentes"]*100
display(hora)

for nome, tabela in {
    "acidentes_por_ano.csv": acidentes_ano,
    "acidentes_por_fase_dia.csv": fase,
    "acidentes_por_uf.csv": uf,
    "tipo_acidente.csv": tipo_acidente,
    "condicao_meteorologica.csv": clima,
    "tipo_pista.csv": tipo_pista,
    "uso_solo.csv": uso_solo,
    "tracado_via.csv": tracado,
    "causas_2021_2025.csv": causas,
    "rodovias.csv": rodovias,
    "hora_dia.csv": hora
}.items():
    tabela.to_csv(os.path.join(PASTA_OUTPUT, nome), index=False, encoding="utf-8-sig")
print("EDA finalizada.")